In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic
import requests
from elasticsearch import Elasticsearch
import elasticsearch
# print(elasticsearch.__version__)  # should show 8.13.0


In [2]:
load_dotenv()
client = Anthropic()

In [4]:
def llm(prompt):
    response = client.messages.create(
        model = "claude-haiku-4-5-20251001"
        ,messages=[{"role": "user", "content": prompt}]
        ,max_tokens=20000
        ,temperature=0
    )
    
    return response.content[0].text

In [43]:
question = 'I just discovered the LLM ZoomCamp course. Can I join now?'
answer = llm(question)
print(answer)

# ZoomCamp LLM Course

I don't have current information about the LLM ZoomCamp course's enrollment status, as details like registration deadlines and cohort schedules change frequently.

To find out if you can join now, I'd recommend:

1. **Visit the official website** - Check DataTalks.Club or the course's main page for current enrollment information
2. **Check their GitHub** - Often has up-to-date details about cohorts and registration
3. **Contact them directly** - Reach out through their official channels to ask about:
   - Current cohort status
   - When the next cohort starts
   - Whether late enrollment is possible
   - Self-paced options (if available)

Many courses allow late joining or offer self-paced alternatives, so it's worth asking even if the current cohort has started.

Is there anything specific about the course content or structure I can help you with?


In [9]:
context = '''
#I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

#I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.
'''


In [11]:

question = 'I just discovered the LLM ZoomCamp course. Can I join now?'

prompt = f''' 
Your task is to answer the questions from course participants based on the provided context. 

Context: {context}
question: {question}
'''

In [47]:
answer = llm(prompt)
print(answer)

Yes, you can still join the LLM Zoomcamp course! 

However, there's an important note: if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.

You can start learning and submitting homework right away without needing to wait for a confirmation email. Registration is optional and is mainly used to gauge interest before the course start date—it's not checked against any registered list.


1. GETTING THE KB FAQ DATABASE VIA REQUESTS

In [5]:
faq_url = 'https://datatalks.club/faq/json/courses.json'
faq_data = requests.get(faq_url).json()

In [6]:
faq_content = []
prefix = 'https://datatalks.club/faq'
for i in faq_data:
    course_url = f'{prefix}{i['path']}'
    course_faq_content = requests.get(course_url).json()
    faq_content.extend(course_faq_content)
len(faq_content)

1342

2. SEARCH

In [8]:
# create index
es = Elasticsearch("http://localhost:9200")

es.options(ignore_status=[400, 404]).indices.delete(index='zoomcamp')
es.options(ignore_status=400).indices.create(
    index='zoomcamp',
    body={
        "mappings": {
            "properties": {
                "question": {"type": "text"},
                "section":  {"type": "text"},
                "answer":   {"type": "text"},
                "course":   {"type": "keyword"}
            }
        }
    }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'zoomcamp'})

In [9]:
# Index documents (equivalent to index.fit(documents))
for doc in faq_content:
    es.index(index='zoomcamp', document=doc)

In [10]:
def search(question, filter_dict={},  boost_dict={}):
    fields =  [f"{k}^{v}" for k, v in boost_dict.items()]  or ["question", "answer", "section"]
    results = es.search(
        index="zoomcamp",
        body={
            "size": 3,
            "query": {
                "bool": {
                    "must":  {"multi_match": {"query": question, "fields": fields}}, #multi_match means "search this query across multiple fields at once."
                    "filter": {"term": filter_dict} if filter_dict else {"match_all": {}} 
                        #filter means exact match on the keyword value which is 'course' defined in this index. Default to match_all
                        #{"term": filter_dict} only works for a single key-value pair.
                }
            }
        }
    )
    return [hit["_source"] for hit in results["hits"]["hits"]]

In [11]:
filter_dict={"course": "llm-zoomcamp"}
boost_dict = {'question':2, 'section':0.5} #search in the question field only. "answer" and "section" are ignored
question = "I just discovered the course. Can I join now?"
search_results_dict = search(question,filter_dict, boost_dict)


In [13]:
boost_dict = {"question": 2.0, "section": 0.5}
boost_dict.items()

dict_items([('question', 2.0), ('section', 0.5)])

In [12]:
[f"{k}^{v}" for k, v in boost_dict.items()]

['question^2', 'section^0.5']

3. PROMPT

In [13]:
# search_results_dict -> relevant_document: process of turning the dictionary format into string's pretty format to feed into LLM
def build_context(search_results_dict):
    lines = []
    for item in search_results_dict:
        section = item['section']
        question = item['question']
        answer = item['answer']
        lines.append(section)
        lines.append('Q: ' + question)
        lines.append('A: ' + answer)
        lines.append('')

    return  '\n'.join(lines).strip()


In [ ]:
def build_prompt(question, search_results_dict):
    relevant_document = build_context(search_results_dict)
    prompt = f'''
Question: 
{question}

Context: 
{relevant_document}
    '''
    return prompt.strip()


In [21]:
question = 'I just discovered the course. Can I join now?'
prompt = build_prompt(question, search_results_dict)
print(prompt)

Question: 
I just discovered the course. Can I join now?

Context: 
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your proje

4. RAG

In [ ]:
def rag(question,instruction, filter_dict, boost_dict):
    search_results_dict = search(question, filter_dict,  boost_dict)
    prompt = build_prompt(question, search_results_dict)

    response = client.messages.create(
        model = "claude-haiku-4-5-20251001" #claude-sonnet-4-6
        ,system = instruction
        ,messages=[{"role": "user", "content": prompt}]
        ,max_tokens=20000
        ,temperature=0
    )
    
    return response.content[0].text

In [ ]:
# question = 'I just discovered the course. Can I join now?'
question = 'How much is the course?'
instruction = '''Your task is to answer the questions from course participants based on the provided context by looking for relevant information to provide accurate answer.
If the answer is not found in the context, respond with "I don't know"'''

filter_dict={"course":"llm-zoomcamp"}
boost_dict ={"question":2, "section":0.5}

print(rag(question, instruction, filter_dict, boost_dict))

Based on the provided context, I don't know the exact cost of the course. The context does not include information about the course price. I would recommend checking the official course website or contacting the course organizers directly for pricing details.
